# 01 — Exploratory Data Analysis: Telco Customer Churn

**Author:** Nicolas Avril · **Dataset:** IBM Telco Customer Churn (7,032 customers × 21 columns)

## Objective
Identify the **drivers of customer churn** in a telecom company and translate them into actionable business levers. Churn rate baseline: ~26.5%.

## Approach
1. Load & sanity-check the cleaned dataset.
2. Profile the target (class balance, base rates).
3. Explore numeric features (tenure, charges) and their relationship to churn.
4. Explore categorical features ranked by signal strength (Cramér's V).
5. Cross-tab the strongest cohort to surface a retention play.

In [ ]:
import sys
from pathlib import Path

# Make local 'churn' package importable when running from notebooks/
ROOT = Path.cwd().parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

from churn.data import load_processed
from churn.config import NUMERIC_FEATURES, CATEGORICAL_FEATURES, TARGET

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110

In [ ]:
df = load_processed()
print(f'Shape: {df.shape}')
print(f'Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')
df.head()

## 1. Target distribution

Churn is **imbalanced** (~26.5% positive class). Implication: accuracy is a poor metric here — we'll lean on **ROC-AUC, PR-AUC, recall and F1** in the modeling notebook.

In [ ]:
churn_rate = df[TARGET].mean()
fig, ax = plt.subplots(figsize=(5, 3.5))
counts = df[TARGET].value_counts().rename({0: 'Stay', 1: 'Churn'})
bars = ax.bar(counts.index, counts.values, color=['#0f766e', '#dc2626'])
for bar, v, p in zip(bars, counts.values, counts.values / counts.sum()):
    ax.text(bar.get_x() + bar.get_width()/2, v + 60, f'{v:,}\n({p:.1%})',
            ha='center', va='bottom', fontsize=10)
ax.set_title(f'Churn distribution — base rate {churn_rate:.1%}')
ax.set_ylabel('Customers')
ax.set_ylim(0, counts.max() * 1.15)
plt.tight_layout(); plt.show()

## 2. Numeric features — distributions split by churn

We expect **tenure** to be the dominant signal: short-tenure customers should churn far more.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, col in zip(axes, ['tenure', 'MonthlyCharges', 'TotalCharges']):
    for label, color in [(0, '#0f766e'), (1, '#dc2626')]:
        sns.kdeplot(df.loc[df[TARGET] == label, col], ax=ax, fill=True,
                    alpha=0.35, color=color, label='Churn' if label else 'Stay')
    ax.set_title(f'{col} by churn status')
    ax.legend()
plt.tight_layout(); plt.show()

**Reading:**
- **Tenure**: churners are concentrated in the first 0–10 months. Customers past the 24-month mark rarely leave. → **Onboarding is the critical retention window.**
- **MonthlyCharges**: churners cluster around \$70–\$100/month. Premium-priced plans correlate with attrition.
- **TotalCharges**: skewed by tenure (mechanical correlation). Less directly informative.

## 3. Categorical features — ranked by Cramér's V

**Cramér's V** measures the strength of association between two categorical variables (0 = independent, 1 = perfect). It's a more honest signal than χ² alone (which is sensitive to sample size).

In [ ]:
def cramers_v(x: pd.Series, y: pd.Series) -> float:
    table = pd.crosstab(x, y)
    chi2 = chi2_contingency(table)[0]
    n = table.sum().sum()
    r, k = table.shape
    return float(np.sqrt(chi2 / (n * (min(k, r) - 1))))

scores = pd.Series(
    {col: cramers_v(df[col], df[TARGET]) for col in CATEGORICAL_FEATURES},
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 6))
scores.plot.barh(ax=ax, color='#2563eb')
ax.set_title("Cramér's V vs Churn — categorical signal strength")
ax.set_xlabel("Cramér's V")
for i, v in enumerate(scores.values):
    ax.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=9)
plt.tight_layout(); plt.show()

**Top drivers (Cramér's V > 0.30):** `Contract`, `OnlineSecurity`, `TechSupport`, `InternetService`. Customer engagement features (security, support) and contract length dominate.

## 4. Contract type × tenure — the killer cross-tab

Contract type is the #1 categorical driver. Let's see how it interacts with tenure.

In [ ]:
df_plot = df.copy()
df_plot['tenure_bucket'] = pd.cut(
    df_plot['tenure'],
    bins=[-1, 6, 12, 24, 48, 72],
    labels=['0-6m', '7-12m', '13-24m', '25-48m', '49-72m'],
)
pivot = (
    df_plot.groupby(['tenure_bucket', 'Contract'], observed=True)[TARGET]
    .mean()
    .unstack()
    .reindex(columns=['Month-to-month', 'One year', 'Two year'])
)

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(pivot * 100, annot=True, fmt='.1f', cmap='Reds', cbar_kws={'label': 'Churn rate (%)'}, ax=ax)
ax.set_title('Churn rate (%) by tenure × contract type')
plt.tight_layout(); plt.show()

**Insight:** the dangerous segment is **month-to-month customers in their first year** (>50% churn rate at 0-6 months). Two-year contracts churn at <3% across all tenures.

**Business lever:** offer aggressive incentives to upgrade month-to-month customers to a 1- or 2-year contract during months 1–6. The expected ROI: every percentage point of conversion in this segment moves the global churn rate by ~0.4pp.

## 5. Internet service × payment method — the silent churners

A subtler interaction: customers with **Fiber optic + Electronic check** payment have abnormal churn — fiber customers are paying premium and the friction of manual electronic checks signals lower commitment.

In [ ]:
pivot2 = (
    df.groupby(['InternetService', 'PaymentMethod'], observed=True)[TARGET]
    .agg(['mean', 'size'])
    .reset_index()
    .rename(columns={'mean': 'churn_rate', 'size': 'n_customers'})
)
pivot2['churn_rate'] = (pivot2['churn_rate'] * 100).round(1)
pivot2.sort_values('churn_rate', ascending=False).head(10)

## 6. Take-aways for modeling (notebook 02)

| Finding | Implication |
|---|---|
| 26.5% churn rate | Use ROC-AUC + PR-AUC, not accuracy. Weight or oversample the minority. |
| Tenure dominates | Engineer `tenure_bucket` and `charges_ratio = TotalCharges / (tenure + 1)` |
| Contract is #1 categorical | Tree-based models will capture interactions; logistic regression needs explicit dummies (already handled by OneHotEncoder). |
| 11 rows had whitespace `TotalCharges` | Already dropped in `churn.data.clean` (0.16%, no signal lost). |
| Class boundaries are largely linear | Logistic regression should be a strong baseline — it was, in fact, the best model in our smoke test (ROC-AUC 0.835). |